# Robustness & Metrics Extraction

This notebook serves as the primary computation engine for the robustness evaluation. It applies five different adversarial attacks (FGSM, PGD, DeepFool, C&W, and Targeted I-FGSM) across three pre-trained models (MobileNetV2, EfficientNetB0, InceptionV3) using a subset of 100 images from MiniImageNet.

**Goal:** Calculate the retained Accuracy, Attack Success Rate (ASR), and average $L_2$ distortion for each model-attack combination.
**Output:** A `robustness_metrics.csv` file that will be used by subsequent visualization notebooks (Radar Charts, Heatmaps, etc.) to quickly plot the results without re-computing the attacks.

**Memory Optimization Note:** To prevent RAM fragmentation (OOM errors) during the thousands of gradient calculations, this script loads models *on-demand*, clearing the TensorFlow backend session and running Garbage Collection exclusively between model transitions. The DeepFool attack has also been optimized to compute all class gradients in a single forward pass using a persistent tape.

*Note: This script performs heavy computations and may take 1-2 hours to complete on a standard GPU.*

In [1]:
import os
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0' # Disable oneDNN to prevent CPU memory fragmentation in long loops
import tensorflow as tf
import numpy as np
import pandas as pd
import glob
import gc

# Import models
from tensorflow.keras.applications import EfficientNetB0, InceptionV3, MobileNetV2

# Import preprocessing and decoding
from tensorflow.keras.applications.efficientnet import preprocess_input as preprocess_eff
from tensorflow.keras.applications.inception_v3 import preprocess_input as preprocess_inc
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as preprocess_mob

In [2]:
# Dictionary with model configurations
models_config = {
    'MobileNetV2': {
        'model_builder': lambda: MobileNetV2(weights='imagenet'),
        'target_size': (224, 224),
        'preprocess_fn': preprocess_mob,
        'clip_min': -1.0,
        'clip_max': 1.0,
        'eps_scale': 1.0,
        'box_min': -1.0,
        'box_max': 1.0
    },
    'EfficientNetB0': {
        'model_builder': lambda: EfficientNetB0(weights='imagenet'),
        'target_size': (224, 224),
        'preprocess_fn': preprocess_eff,
        'clip_min': 0.0,
        'clip_max': 255.0,
        'eps_scale': 127.5,
        'box_min': 0.0,
        'box_max': 255.0
    },
    'InceptionV3': {
        'model_builder': lambda: InceptionV3(weights='imagenet'),
        'target_size': (299, 299),
        'preprocess_fn': preprocess_inc,
        'clip_min': -1.0,
        'clip_max': 1.0,
        'eps_scale': 1.0,
        'box_min': -1.0,
        'box_max': 1.0
    }
}

# Instead of instantiating all 3 models at once, we store lambda functions
# This ensures only one model is loaded in memory at any given time

### Helper Functions & Data Loading

In [3]:
def load_and_preprocess_image(img_path, target_size, preprocess_fn):
    img_raw = tf.io.read_file(img_path)
    img = tf.image.decode_image(img_raw, channels=3)
    img = tf.cast(img, tf.float32)
    img = tf.image.resize(img, target_size)
    img = preprocess_fn(img) # Dynamic preprocessing
    img = tf.expand_dims(img, axis=0) # Add batch dimension
    return img

In [4]:
# Define the path to the 100 images folder
dataset_path = '../images/miniimagenet_random_100/'
image_paths = glob.glob(os.path.join(dataset_path, '*.*'))

# Limit to 100 just in case
image_paths = image_paths[:100]
print(f"Total images loaded for the experiment: {len(image_paths)}")

Total images loaded for the experiment: 100


### Attack Implementations
We define the 5 attacks here. We use standardized hyperparameters to ensure a fair comparison across models.

In [5]:
loss_object = tf.keras.losses.CategoricalCrossentropy()

In [6]:
def fgsm_attack(img, label, epsilon, model, clip_min, clip_max):
    with tf.GradientTape() as tape:
        tape.watch(img)
        pred = model(img, training=False)
        loss = loss_object(label, pred)
    grad = tape.gradient(loss, img)
    return tf.clip_by_value(img + epsilon * tf.sign(grad), clip_min, clip_max)

In [7]:
def pgd_attack(img, label, epsilon, model, clip_min, clip_max, iters=10):
    alpha = epsilon / (iters / 2.0)
    adv_img = tf.identity(img)
    for _ in range(iters):
        with tf.GradientTape() as tape:
            tape.watch(adv_img)
            pred = model(adv_img, training=False)
            loss = loss_object(label, pred)
        grad = tape.gradient(loss, adv_img)
        adv_img = adv_img + alpha * tf.sign(grad)
        perturbation = tf.clip_by_value(adv_img - img, -epsilon, epsilon)
        adv_img = tf.clip_by_value(img + perturbation, clip_min, clip_max)
    return adv_img

In [8]:
def cw_attack(img, label, model, box_min, box_max, c_weight=1.0, max_iters=40, lr=0.05):
    # Memory optimization: Manual Gradient Descent
    modifier = tf.zeros_like(img)
    
    for _ in range(max_iters):
        with tf.GradientTape() as tape:
            tape.watch(modifier) # Watch the primitive tensor
            adv_norm = 0.5 * (tf.tanh(modifier) + 1.0)
            adv_img = adv_norm * (box_max - box_min) + box_min
            
            l2_loss = tf.reduce_sum(tf.square(adv_img - img))
            preds = model(adv_img, training=False)
            
            real_prob = tf.reduce_sum(label * preds, axis=1)
            other_prob = tf.reduce_max((1.0 - label) * preds, axis=1)
            f_loss = tf.maximum(0.0, real_prob - other_prob)
            total_loss = l2_loss + c_weight * f_loss
            
        grads = tape.gradient(total_loss, modifier)
        # Manual parameter update (Standard Gradient Descent)
        modifier = modifier - lr * grads
        
    adv_norm = 0.5 * (tf.tanh(modifier) + 1.0)
    return adv_norm * (box_max - box_min) + box_min

In [9]:
def deepfool_attack(img, model, clip_min, clip_max, num_classes=10, overshoot=0.02, max_iter=20):
    adv_img = tf.identity(img)
    top_classes = tf.argsort(model(adv_img, training=False)[0], direction='DESCENDING')[:num_classes]
    orig_label = tf.cast(top_classes[0], tf.int32)
    curr_label = orig_label
    iteration = 0
    
    while curr_label == orig_label and iteration < max_iter:
        perturbation = tf.zeros_like(adv_img)
        min_w_norm = float('inf')
        
        with tf.GradientTape(persistent=True) as tape:
            tape.watch(adv_img)
            preds = model(adv_img, training=False)[0]
            orig_loss = preds[orig_label]
            
            target_losses = []
            for k in range(1, num_classes):
                t_class = tf.cast(top_classes[k], tf.int32)
                target_losses.append(preds[t_class])
                
        orig_grad = tape.gradient(orig_loss, adv_img)
        
        for i in range(num_classes - 1):
            target_grad = tape.gradient(target_losses[i], adv_img)
            
            w_k = target_grad - orig_grad
            f_k = target_losses[i] - orig_loss
            w_norm = tf.norm(w_k)
            if w_norm == 0: continue
            
            distance = tf.abs(f_k) / (w_norm + 1e-6)
            if distance < min_w_norm:
                min_w_norm = distance
                perturbation = (distance * w_k) / (w_norm + 1e-6)
                
        # Explicitly free the persistent tape
        del tape 
        
        adv_img = tf.clip_by_value(adv_img + (1 + overshoot) * perturbation, clip_min, clip_max)
        curr_label = tf.cast(tf.argmax(model(adv_img, training=False)[0]), tf.int32)
        iteration += 1
        
    return adv_img

In [10]:
def targeted_ifgsm_attack(img, target_label, epsilon, model, clip_min, clip_max, iters=20):
    alpha = epsilon / (iters / 2.0)
    adv_img = tf.identity(img)
    for _ in range(iters):
        with tf.GradientTape() as tape:
            tape.watch(adv_img)
            pred = model(adv_img, training=False)
            loss = loss_object(target_label, pred)
        grad = tape.gradient(loss, adv_img)
        adv_img = adv_img - alpha * tf.sign(grad)
        perturbation = tf.clip_by_value(adv_img - img, -epsilon, epsilon)
        adv_img = tf.clip_by_value(img + perturbation, clip_min, clip_max)
    return adv_img

### Global Evaluation Loop
We evaluate each model against all attacks. We assume a baseline $\varepsilon=0.05$ (scaled accordingly) and a target class `301` (Ladybug) for the targeted attack.

In [11]:
BASE_EPSILON = 0.05
TARGET_CLASS = 301 # Ladybug
RAW_CSV = 'robustness_metrics_raw.csv'

# Checkpoint system
processed_tasks = set()
if os.path.exists(RAW_CSV):
    df_prog = pd.read_csv(RAW_CSV)
    for _, row in df_prog.iterrows():
        processed_tasks.add((row['Model'], row['Image_ID']))
    print(f"Resuming progress: Found {len(processed_tasks)} processed images.")
else:
    with open(RAW_CSV, 'w') as f:
        f.write("Model,Image_ID,Attack,Is_Correct,Is_Success,L2_Distance\n")

# Evaluation loop
for model_name, config in models_config.items():
    print(f"\n{'='*60}\nEvaluating Model: {model_name}\n{'='*60}")
    
    tf.keras.backend.clear_session()
    gc.collect()
    model = config['model_builder']()
    model.trainable = False
    
    eps = BASE_EPSILON * config['eps_scale']
    c_min, c_max = config['clip_min'], config['clip_max']
    
    for i, img_path in enumerate(image_paths):
        # Skip previously processed images
        if (model_name, i) in processed_tasks:
            continue
            
        print(f"Processing image {i+1}/100 for {model_name}...")
        
        # Memory cleanup: Rebuild model every 5 images
        if i % 5 == 0 and i > 0:
            del model
            tf.keras.backend.clear_session()
            gc.collect()
            model = config['model_builder']()
            model.trainable = False
            
        img = load_and_preprocess_image(img_path, config['target_size'], config['preprocess_fn'])
        
        orig_preds = model(img, training=False)
        orig_idx = tf.argmax(orig_preds, axis=-1).numpy()[0]
        
        one_hot_label = tf.reshape(tf.one_hot(orig_idx, orig_preds.shape[-1]), (1, -1))
        target_one_hot = tf.reshape(tf.one_hot(TARGET_CLASS, orig_preds.shape[-1]), (1, -1))
        
        img_metrics = {
            'Baseline': {'correct': 1, 'success': 0, 'l2': 0.0},
            'FGSM':     {'correct': 0, 'success': 0, 'l2': 0.0},
            'PGD':      {'correct': 0, 'success': 0, 'l2': 0.0},
            'DeepFool': {'correct': 0, 'success': 0, 'l2': 0.0},
            'C&W':      {'correct': 0, 'success': 0, 'l2': 0.0},
            'T-IFGSM':  {'correct': 0, 'success': 0, 'l2': 0.0}
        }
        
        def evaluate_attack(attack_name, adv_img, targeted=False):
            adv_preds = model(adv_img, training=False)
            adv_idx = tf.argmax(adv_preds, axis=-1).numpy()[0]
            
            img_metrics[attack_name]['l2'] = float(np.linalg.norm(img.numpy() - adv_img.numpy()))
            
            if adv_idx == orig_idx:
                img_metrics[attack_name]['correct'] = 1 
            if targeted:
                if adv_idx == TARGET_CLASS:
                    img_metrics[attack_name]['success'] = 1 
            else:
                if adv_idx != orig_idx:
                    img_metrics[attack_name]['success'] = 1 
                    
        # FGSM
        adv_fgsm = fgsm_attack(img, one_hot_label, eps, model, c_min, c_max)
        evaluate_attack('FGSM', adv_fgsm)
        del adv_fgsm; gc.collect()

        # PGD
        adv_pgd = pgd_attack(img, one_hot_label, eps, model, c_min, c_max, iters=10)
        evaluate_attack('PGD', adv_pgd)
        del adv_pgd; gc.collect()

        # C&W
        adv_cw = cw_attack(img, one_hot_label, model, config['box_min'], config['box_max'], max_iters=40)
        evaluate_attack('C&W', adv_cw)
        del adv_cw; gc.collect()

        # DeepFool
        adv_df = deepfool_attack(img, model, c_min, c_max, max_iter=20)
        evaluate_attack('DeepFool', adv_df)
        del adv_df; gc.collect()

        # Targeted I-FGSM
        adv_tifgsm = targeted_ifgsm_attack(img, target_one_hot, eps, model, c_min, c_max, iters=20)
        evaluate_attack('T-IFGSM', adv_tifgsm, targeted=True)
        del adv_tifgsm; gc.collect()
        
        # Store in CSV
        with open(RAW_CSV, 'a') as f:
            for atk, vals in img_metrics.items():
                f.write(f"{model_name},{i},{atk},{vals['correct']},{vals['success']},{vals['l2']}\n")
                
        # Clean local variables
        del img, orig_preds, one_hot_label, target_one_hot
        gc.collect()
        
    del model
    gc.collect()

print("\nAll Models Evaluated Successfully!")

Resuming progress: Found 286 processed images.

Evaluating Model: MobileNetV2


Evaluating Model: EfficientNetB0

Evaluating Model: InceptionV3
Processing image 87/100 for InceptionV3...
Processing image 88/100 for InceptionV3...
Processing image 89/100 for InceptionV3...
Processing image 90/100 for InceptionV3...
Processing image 91/100 for InceptionV3...
Processing image 92/100 for InceptionV3...
Processing image 93/100 for InceptionV3...
Processing image 94/100 for InceptionV3...
Processing image 95/100 for InceptionV3...
Processing image 96/100 for InceptionV3...
Processing image 97/100 for InceptionV3...
Processing image 98/100 for InceptionV3...
Processing image 99/100 for InceptionV3...
Processing image 100/100 for InceptionV3...

All Models Evaluated Successfully!


### Export Results to CSV
We save the aggregated metrics into a DataFrame and export it. Subsequent notebooks will load this file directly.

In [12]:
# Load raw data
df_raw = pd.read_csv('robustness_metrics_raw.csv')

df_summary = df_raw.groupby(['Model', 'Attack']).mean().reset_index()

df_summary['Accuracy (%)'] = df_summary['Is_Correct'] * 100
df_summary['ASR (%)'] = df_summary['Is_Success'] * 100
df_summary['Avg_L2'] = df_summary['L2_Distance']

df_final = df_summary[['Model', 'Attack', 'Accuracy (%)', 'ASR (%)', 'Avg_L2']]

display(df_final)

# Save to CSV
output_filename = 'robustness_metrics.csv'
df_final.to_csv(output_filename, index=False)
print(f"Aggregated metrics successfully saved to '{output_filename}'")

,Model,Attack,Accuracy (%),ASR (%),Avg_L2
0,EfficientNetB0,Baseline,100.0,0.0,0.000000
1,EfficientNetB0,C&W,35.0,65.0,29149.570449
2,EfficientNetB0,DeepFool,3.0,97.0,106.820442
3,EfficientNetB0,FGSM,8.0,92.0,2440.111860
4,EfficientNetB0,PGD,0.0,100.0,1401.132029
5,EfficientNetB0,T-IFGSM,1.0,99.0,1218.856450
6,InceptionV3,Baseline,100.0,0.0,0.000000
7,InceptionV3,C&W,7.0,93.0,35.850081
8,InceptionV3,DeepFool,5.0,95.0,2.146886
9,InceptionV3,FGSM,9.0,91.0,25.546868


Aggregated metrics successfully saved to 'robustness_metrics.csv'
